# COMP47590 Advanced Machine Learning

## Assignment: Learning to Walk
Uses the PPO actor-critic method to train a neural network to allow a simple robot to learn to walk using the **Bipedal Walker** environment from the `gymnasium` package (https://gymnasium.farama.org/environments/box2d/bipedal_walker/). This assignment uses a custom version of the environment that includes some small modifications, `BipedalWalkerUCD`.

![Bipedal Walker](bipedal_walker.gif)

The **action** space is continuous where the actions are motor speed values in the [-1, 1] range for each of the 4 joints at both hips and knees.

The **observation** is a state vector consisting of hull angle speed, angular velocity, horizontal speed, vertical speed, position of joints and joints angular speed, legs contact with ground, and 10 lidar rangefinder measurements. There are no coordinates in the state vector.

The basic **reward** provides a small positive rewqard for every step forwards (left to right) the agent takes.

### Initialisation

If using Google colab you need to install packages - comment out lines below.

In [ ]:
#!apt install swig cmake ffmpeg
#!apt-get install -y xvfb x11-utils
#!pip install stable-baselines3[extra] pyglet box2d box2d-kengz
#!pip install pyvirtualdisplay PyOpenGL PyOpenGL-accelerate

Import required packages.

In [1]:
import gymnasium as gym
import stable_baselines3 as sb3
import torch

import bipedal_walker_ucd

import os
import pandas as pd # For data frames and data frame manipulation
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
import numpy as np # For general  numeric operations
from collections import deque

import gc # Import garbage collector

import imageio

import matplotlib.pyplot as plt
%matplotlib inline

### Define a Custom Wrapper and Evaluation Callback Function

We define a custom wrapper to allow experimantion twith different rewards in the Bipedal Walker environment. This adds potential extra reward terms:

- **Stability:** Encourage the hull to stay horizontal.
- **Target Speed:** Encaourage the agent to move at a target speed. The target speed can be set.
- **Smoothness:** Encourages similar action from one step to the next.

Each of these different com-oentnts of the reward can have a different weight associated with it. These are passed as parameters to the reward wrapper.
- `action_penalty_weight`
- `stability_weight`
- `speed_weight`
- `smoothness_weight`

When speed is given a non-zero weight a target speed should also be provided as `target_speed`.

In [2]:
class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, window=20,
                 action_penalty_weight = 0.0,
                 stability_weight=0.0,
                 smoothness_weight = 0.0,
                 speed_weight=0.0,
                 target_speed=1.5
                 ):
        super().__init__(env)
        self.window = window
        self.action_penalty_weight = action_penalty_weight
        self.stability_weight = stability_weight
        self.smoothness_weight = smoothness_weight
        self.speed_weight = speed_weight
        self.target_speed = target_speed

        self.prev_action = np.zeros(env.action_space.shape)
        self.hip1_history = deque(maxlen=window)
        self.hip2_history = deque(maxlen=window)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)

        # --- Extract state ---
        hull = self.env.unwrapped.hull
        hull_angle   = hull.angle
        hull_vel_x   = hull.linearVelocity[0]
        hull_vel_y   = hull.linearVelocity[1]
        hull_height  = hull.position.y

        hip1_angle  = obs[4]
        hip2_angle  = obs[10]
        hip1_vel    = obs[5]
        hip2_vel    = obs[11]
        knee1_angle = obs[6]
        knee2_angle = obs[12]
        knee1_vel   = obs[7]
        knee2_vel   = obs[13]

        # --- Action Penalty ----
        # normalized to about -50.0 using heuristic, more optimal agent should spend less
        action_penalty = 0
        for a in action:
            action_penalty -= 0.00035 * bipedal_walker_ucd.MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
        action_penalty = action_penalty*self.action_penalty_weight

        # --- Stability ---
        stability = -abs(hull_angle) * self.stability_weight

        # --- Smoothness ---
        smoothness = -np.sum(np.abs(action - self.prev_action)) * self.smoothness_weight

        # --- Speed targeting ---
        speed_shaping = -abs(hull_vel_x - self.target_speed) * self.speed_weight

        # --- Total reward ---
        modified_reward = (
            base_reward
            + action_penalty
            + stability
            + smoothness
            + speed_shaping
        )

        # --- Reward components for TensorBoard ---
        info["reward_components"] = {
            "base":              base_reward,
            "action_penalty":    action_penalty,
            "stability":         stability,
            "smoothness":        smoothness,
            "speed_shaping":     speed_shaping,
            "total":             modified_reward,
        }

        self.prev_action = action.copy()
        return obs, modified_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.prev_action = np.zeros(self.env.action_space.shape)
        self.hip1_history.clear()
        self.hip2_history.clear()
        return self.env.reset(**kwargs)

The `VideoEvalCallback` writes out the components of the reward function defined above and also records videos of agent behaviour.

In [3]:

class VideoEvalCallback(sb3.common.callbacks.BaseCallback):
    def __init__(
        self,
        eval_env: gym.Env,
        eval_freq: int = 10_000,
        n_eval_episodes: int = 1,
        video_folder: str = "./eval_videos",
        video_length: int = 1000,
        tensorboard_log: str = "./tb_logs",
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.video_folder = video_folder
        self.video_length = video_length
        self.writer = torch.utils.tensorboard.SummaryWriter(log_dir=tensorboard_log)
        self.eval_count = 0
        os.makedirs(video_folder, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            self._run_eval()
        return True

    def _run_eval(self):
        self.eval_count += 1
        episode_rewards = []
        video_frames = None  # only keep episode 0 frames
        component_totals = {}

        for episode in range(self.n_eval_episodes):
            obs, _ = self.eval_env.reset()
            done = False
            total_reward = 0.0
            ep_frames = []
            ep_components = {}

            while not done:
                # Only render for episode 0 (video episode)
                if episode == 0:
                    frame = self.eval_env.render()
                    if frame is not None:
                        ep_frames.append(frame)

                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.eval_env.step(action)
                total_reward += reward
                done = terminated or truncated

                components = info.get("reward_components", {})
                for key, val in components.items():
                    ep_components[key] = ep_components.get(key, 0.0) + val

            episode_rewards.append(total_reward)

            for key, val in ep_components.items():
                component_totals[key] = component_totals.get(key, 0.0) + val

            if episode == 0:
                video_frames = ep_frames

            # Explicitly free ep_frames for non-video episodes
            del ep_frames

        mean_reward = np.mean(episode_rewards)
        std_reward  = np.std(episode_rewards)

        self.writer.add_scalar("eval/mean_reward", mean_reward, self.num_timesteps)
        self.writer.add_scalar("eval/std_reward",  std_reward,  self.num_timesteps)

        for key, total in component_totals.items():
            mean_val = total / self.n_eval_episodes
            self.writer.add_scalar(
                f"eval/reward_components/{key}", mean_val, self.num_timesteps
            )

        if self.verbose:
            print(f"\n[Eval @ {self.num_timesteps} steps]")
            print(f"  Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
            for key, total in component_totals.items():
                print(f"    {key:>20s}: {total / self.n_eval_episodes:.4f}")

        if video_frames:
            self._log_video_to_tensorboard(video_frames)

        # Free video frames after logging
        del video_frames
        gc.collect()

    def _log_video_to_tensorboard(self, frames: list):
        video_path = os.path.join(
            self.video_folder, f"eval_{self.eval_count}_step_{self.num_timesteps}.mp4"
        )
        imageio.mimsave(video_path, frames, fps=30)

        if self.verbose:
            print(f"  Video saved to {video_path}")

        #video_array = np.array(frames, dtype=np.uint8)       # (T, H, W, C)
        #video_array = video_array.transpose(0, 3, 1, 2)      # (T, C, H, W)
        #video_array = np.expand_dims(video_array, axis=0)    # (1, T, C, H, W)
        #video_tensor = video_array.astype(np.float32) / 255.0
#
        #self.writer.add_video(
        #    "eval/agent_behaviour",
        #    video_tensor,
        #    global_step=self.num_timesteps,
        #    fps=30,
        #)
        #self.writer.flush()

        # Free the large arrays immediately
        #del video_array, video_tensor

    def _on_training_end(self):
        self.writer.close()

In [3]:

class VideoEvalCallback(sb3.common.callbacks.BaseCallback):
    def __init__(
        self,
        eval_env: gym.Env,
        eval_freq: int = 10_000,
        n_eval_episodes: int = 1,
        video_folder: str = "./eval_videos",
        video_length: int = 500,
        tensorboard_log: str = "./tb_logs",
        verbose: int = 1,
    ):
        super().__init__(verbose)
        self.eval_env = eval_env
        self.eval_freq = eval_freq
        self.n_eval_episodes = n_eval_episodes
        self.video_folder = video_folder
        self.video_length = video_length
        self.writer = torch.utils.tensorboard.SummaryWriter(log_dir=tensorboard_log)
        self.eval_count = 0
        os.makedirs(video_folder, exist_ok=True)

    def _on_step(self) -> bool:
        if self.n_calls % self.eval_freq == 0:
            self._run_eval()
        return True

    def _run_eval(self):
        self.eval_count += 1
        episode_rewards = []
        video_frames = None  # only keep episode 0 frames
        component_totals = {}

        for episode in range(self.n_eval_episodes):
            obs, _ = self.eval_env.reset()
            done = False
            total_reward = 0.0
            ep_frames = []
            ep_components = {}
            step_count = 0

            while not done:
                # Only render for episode 0, and cap frames to video_length
                if episode == 0 and step_count < self.video_length:
                    frame = self.eval_env.render()
                    if frame is not None:
                        ep_frames.append(frame)

                action, _ = self.model.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = self.eval_env.step(action)
                total_reward += reward
                done = terminated or truncated
                step_count += 1

                components = info.get("reward_components", {})
                for key, val in components.items():
                    ep_components[key] = ep_components.get(key, 0.0) + val

            episode_rewards.append(total_reward)

            for key, val in ep_components.items():
                component_totals[key] = component_totals.get(key, 0.0) + val

            if episode == 0:
                video_frames = ep_frames

            # Explicitly free ep_frames for non-video episodes
            del ep_frames

        mean_reward = np.mean(episode_rewards)
        std_reward  = np.std(episode_rewards)

        self.writer.add_scalar("eval/mean_reward", mean_reward, self.num_timesteps)
        self.writer.add_scalar("eval/std_reward",  std_reward,  self.num_timesteps)

        for key, total in component_totals.items():
            mean_val = total / self.n_eval_episodes
            self.writer.add_scalar(
                f"eval/reward_components/{key}", mean_val, self.num_timesteps
            )

        if self.verbose:
            print(f"\n[Eval @ {self.num_timesteps} steps]")
            print(f"  Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")
            for key, total in component_totals.items():
                print(f"    {key:>20s}: {total / self.n_eval_episodes:.4f}")

        if video_frames:
            self._save_video(video_frames)

        # Free video frames after logging and force garbage collection
        del video_frames
        gc.collect()

    def _save_video(self, frames: list):
        video_path = os.path.join(
            self.video_folder, f"eval_{self.eval_count}_step_{self.num_timesteps}.mp4"
        )
        imageio.mimsave(video_path, frames, fps=30)

        if self.verbose:
            print(f"  Video saved to {video_path}")

        # Log only a scalar indicator instead of the full video tensor to avoid RAM spike
        self.writer.add_scalar("eval/video_saved", self.eval_count, self.num_timesteps)
        self.writer.flush()

    def _on_training_end(self):
        self.writer.close()


### Task 1 Run A Basic Experiment
Create an instance of the **BipedalWalkerUCD** environment and add a `CustomRewardWrapper` defining the reward mix that will be used in the experiment - in this case set the weight of all non-base components to zero.

#### Create the Training Environment
Weight components are defined here for reuse throughout the notebook. It is also worth adding a `TimeLimt` wrapper to avoid very long runs during training. All weights are set to `0.0` for this experiment.

In [5]:
action_penalty_weight = 0
symmetry_weight = 0
stability_weight = 0
smoothness_weight = 0
speed_weight = 0
target_speed = 1.5

We also define a set of utility values used throughout the experiment.

In [4]:
time_limit = 5000
tb_log_path = './log_tb_bipedal_PPO/'
eval_log_path = './logs_bipedal_PPO/'
run_name = "Basic_Reward"
eval_video_path = './eval_videos/' + run_name + '/'
tb_log_name = run_name

Create the environment.

In [6]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode=None)
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                         )
env = gym.wrappers.TimeLimit(env,
                                max_episode_steps = time_limit)

Explore the environment - view the action space and observation space.

In [7]:
env.action_space

Box(-1.0, 1.0, (4,), float32)

In [8]:
env.observation_space

Box([-3.1415927 -5.        -5.        -5.        -3.1415927 -5.
 -3.1415927 -5.        -0.        -3.1415927 -5.        -3.1415927
 -5.        -0.        -1.        -1.        -1.        -1.
 -1.        -1.        -1.        -1.        -1.        -1.       ], [3.1415927 5.        5.        5.        3.1415927 5.        3.1415927
 5.        5.        3.1415927 5.        3.1415927 5.        5.
 1.        1.        1.        1.        1.        1.        1.
 1.        1.        1.       ], (24,), float32)

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [15]:
agent = sb3.PPO('MlpPolicy',
                env,
                batch_size = 64,
                clip_range = 0.2,
                ent_coef = 0.001,
                gae_lambda =  0.95,
                gamma = 0.99,
                learning_rate = 0.00025,
                n_steps = 2048,
                n_epochs = 10,
                verbose = 2,
                tensorboard_log = tb_log_path
                )

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\ABHIDYU\AppData\Local\Programs\Python\Python313\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


Examine the actor and critic network architectures.

In [16]:
print(agent.policy)

ActorCriticPolicy(
  (features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (pi_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (vf_features_extractor): FlattenExtractor(
    (flatten): Flatten(start_dim=1, end_dim=-1)
  )
  (mlp_extractor): MlpExtractor(
    (policy_net): Sequential(
      (0): Linear(in_features=24, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
    (value_net): Sequential(
      (0): Linear(in_features=24, out_features=64, bias=True)
      (1): Tanh()
      (2): Linear(in_features=64, out_features=64, bias=True)
      (3): Tanh()
    )
  )
  (action_net): Linear(in_features=64, out_features=4, bias=True)
  (value_net): Linear(in_features=64, out_features=1, bias=True)
)


#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [17]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode='rgb_array')
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                         )
eval_env = gym.wrappers.TimeLimit(eval_env,
                                max_episode_steps = time_limit)

eval_log_path = eval_log_path

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = 5000,        # evaluate every 20k steps
    n_eval_episodes = 1,       # average reward over 3 episodes, video from first
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose=1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [13]:
agent.learn(total_timesteps = 500000,
            callback = video_callback,
            tb_log_name = run_name)

Logging to ./log_tb_bipedal_PPO/Basic_Reward_3
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 63.4     |
|    ep_rew_mean     | -95.5    |
| time/              |          |
|    fps             | 730      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 63.4         |
|    ep_rew_mean          | -95.5        |
| time/                   |              |
|    fps                  | 614          |
|    iterations           | 2            |
|    time_elapsed         | 6            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0062917816 |
|    clip_fraction        | 0.0381       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.68        |
|    explai

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).



[Eval @ 5000 steps]
  Mean reward: -89.04 +/- 0.00
                    base: -89.0433
          action_penalty: 0.0000
               stability: 0.0000
              smoothness: 0.0000
           speed_shaping: 0.0000
                   total: -89.0433
  Video saved to ./eval_videos/Basic_Reward/eval_1_step_5000.mp4
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 689         |
|    ep_rew_mean          | -80.5       |
| time/                   |             |
|    fps                  | 547         |
|    iterations           | 3           |
|    time_elapsed         | 11          |
|    total_timesteps      | 6144        |
| train/                  |             |
|    approx_kl            | 0.009287285 |
|    clip_fraction        | 0.133       |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.65       |
|    explained_variance   | 0.395       |
|    learning_rate        | 0.00025     |
|    loss

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).



[Eval @ 10000 steps]
  Mean reward: -91.87 +/- 0.00
                    base: -91.8714
          action_penalty: 0.0000
               stability: 0.0000
              smoothness: 0.0000
           speed_shaping: 0.0000
                   total: -91.8714
  Video saved to ./eval_videos/Basic_Reward/eval_2_step_10000.mp4
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 908         |
|    ep_rew_mean          | -84.9       |
| time/                   |             |
|    fps                  | 522         |
|    iterations           | 5           |
|    time_elapsed         | 19          |
|    total_timesteps      | 10240       |
| train/                  |             |
|    approx_kl            | 0.009001896 |
|    clip_fraction        | 0.068       |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.63       |
|    explained_variance   | 0.125       |
|    learning_rate        | 0.00025     |
|    lo

...


[Eval @ 490000 steps]
  Mean reward: -82.92 +/- 0.00
                    base: -82.9195
          action_penalty: 0.0000
               stability: 0.0000
              smoothness: 0.0000
           speed_shaping: 0.0000
                   total: -82.9195
  Video saved to ./eval_videos/Basic_Reward/eval_98_step_490000.mp4
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.29e+03    |
|    ep_rew_mean          | 233         |
| time/                   |             |
|    fps                  | 405         |
|    iterations           | 240         |
|    time_elapsed         | 1212        |
|    total_timesteps      | 491520      |
| train/                  |             |
|    approx_kl            | 0.009609345 |
|    clip_fraction        | 0.134       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.67       |
|    explained_variance   | 0.25        |
|    learning_rate        | 0.00025     |
|   

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).



[Eval @ 495000 steps]
  Mean reward: -37.82 +/- 0.00
                    base: -37.8199
          action_penalty: 0.0000
               stability: 0.0000
              smoothness: 0.0000
           speed_shaping: 0.0000
                   total: -37.8199
  Video saved to ./eval_videos/Basic_Reward/eval_99_step_495000.mp4
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.3e+03     |
|    ep_rew_mean          | 237         |
| time/                   |             |
|    fps                  | 405         |
|    iterations           | 242         |
|    time_elapsed         | 1222        |
|    total_timesteps      | 495616      |
| train/                  |             |
|    approx_kl            | 0.012092618 |
|    clip_fraction        | 0.153       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.68       |
|    explained_variance   | 0.319       |
|    learning_rate        | 0.00025     |
|   

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (600, 400) to (608, 400) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).



[Eval @ 500000 steps]
  Mean reward: 82.98 +/- 0.00
                    base: 82.9775
          action_penalty: 0.0000
               stability: 0.0000
              smoothness: 0.0000
           speed_shaping: 0.0000
                   total: 82.9775
  Video saved to ./eval_videos/Basic_Reward/eval_100_step_500000.mp4
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.31e+03    |
|    ep_rew_mean          | 239         |
| time/                   |             |
|    fps                  | 405         |
|    iterations           | 245         |
|    time_elapsed         | 1238        |
|    total_timesteps      | 501760      |
| train/                  |             |
|    approx_kl            | 0.007297918 |
|    clip_fraction        | 0.0587      |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.69       |
|    explained_variance   | 0.9         |
|    learning_rate        | 0.00025     |
|    l

Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal_PPO/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### Save the Agent
Save the trained agent.

In [14]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [15]:
del agent
env.close()
del env
eval_env.close()
del eval_env
gc.collect()

55

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

#### Task 1 Reflection 

Training with only the base reward was a useful starting point, even if the results weren't pretty. Early on the agent was essentially useless. The first evaluation video showed it collapsing almost immediately, scoring around −100. It took until roughly 150k steps before anything interesting emerged: the agent figured out that throwing its weight forward before falling still counts as forward progress, so it started doing these short, lurching bursts rather than anything resembling a walk.

By 200–300k steps it had settled into a repeatable strategy: kick one leg hard, crash forward, repeat. Not elegant, but it worked. The final video at 500k steps showed the agent covering the full terrain with a total reward close to 300, which was genuinely surprising to watch given how chaotic the motion looked.

The gait that emerged is basically all torque, all the time. Since the reward only cares about moving right, the agent never learned to be efficient or smooth. It just hammered the motors as hard as possible on every step, finding the easiest path to a high reward, which happens to look nothing like actual walking.

The training curve reflected this. Early episodes were all over the place because the −100 fall penalty dominated, meaning the agent had to first learn not to die before it could learn to move. You could see this in the ep_len_mean metric creeping up from around 74 steps to the full 5,000, a clear sign the agent was surviving longer before falling. PPO kept things stable throughout, but without any shaping on energy or posture, the policy had no reason to look physically realistic. It's a good baseline precisely because it shows what happens when you give a learning agent a goal but no constraints on how to achieve it.

### Task 2 Run A Custom Reward Experiment
Run an experiment using a version of the cutoms reward function with some of the extra compenets turned on to create a smoother gait.

#### Create the Training Environment
Weight components are defined here for reuse throughout the notebook. It is also worth adding a `TimeLimt` wrapper to avoid very long runs during training. Add non-zero weight values for some of thes components (values between 0.0 and 2.0 will work well).

In [5]:
action_penalty_weight = 3.0   # discourages wasteful full-torque bursts
stability_weight      = 1.5   # penalises hull tilt -> upright posture
smoothness_weight     = 0.3   # penalises large step-to-step action changes
speed_weight          = 0.5   # soft pull toward a moderate walking pace
target_speed          = 1.5

We also define a set of utility values used throughout the experiment.

In [7]:
time_limit = 5000
tb_log_path = './log_tb_bipedal_PPO/'
eval_log_path = './logs_bipedal_PPO/'
run_name = 'Task2_Smooth_Gait'
eval_video_path = './eval_videos/' + run_name + '/'
tb_log_name = run_name

Create the environment.

In [8]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode=None)
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed)
env = gym.wrappers.TimeLimit(env, max_episode_steps=time_limit)

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [10]:
agent = sb3.PPO('MlpPolicy',
                env,
                batch_size = 64,
                clip_range = 0.2,
                ent_coef = 0.001,
                gae_lambda =  0.95,
                gamma = 0.99,
                learning_rate = 0.00025,
                n_steps = 2048,
                n_epochs = 10,
                verbose = 2,
                tensorboard_log = tb_log_path
                )

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\ABHIDYU\AppData\Local\Programs\Python\Python313\Lib\site-packages\stable_baselines3\common\on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [11]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode='rgb_array')
eval_env = CustomRewardWrapper(eval_env,
                               action_penalty_weight = action_penalty_weight,
                               stability_weight      = stability_weight,
                               smoothness_weight     = smoothness_weight,
                               speed_weight          = speed_weight,
                               target_speed          = target_speed)
eval_env = gym.wrappers.TimeLimit(eval_env, max_episode_steps=time_limit)

video_callback = VideoEvalCallback(
    eval_env        = eval_env,
    eval_freq       = 50_000,   # 10 checkpoints over a 500k-step run
    n_eval_episodes = 1,        # reliable mean ± std across 3 episodes
    video_folder    = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose         = 1,
)


#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [ ]:
agent.learn(total_timesteps=500000,
            callback=video_callback,
            tb_log_name=run_name,
            reset_num_timesteps=True,
            progress_bar=True)

Logging to ./log_tb_bipedal_PPO/Task2_Smooth_Gait_4


c:\Users\ABHIDYU\AppData\Local\Programs\Python\Python313\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 87       |
|    ep_rew_mean     | -383     |
| time/              |          |
|    fps             | 728      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 87           |
|    ep_rew_mean          | -383         |
| time/                   |              |
|    fps                  | 619          |
|    iterations           | 2            |
|    time_elapsed         | 6            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0064343987 |
|    clip_fraction        | 0.0664       |
|    clip_range           | 0.2          |
|    entropy_loss         | -5.68        |
|    explained_variance   | 0.0138       |
|    learning_r

c:\Users\ABHIDYU\AppData\Local\Programs\Python\Python313\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal_PPO/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### Save the Agent
Save the trained agent.

In [12]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [13]:
del agent
env.close()
del env
eval_env.close()
del eval_env
gc.collect()

125

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

#### Task 2 Reflection 

Adding the four shaping components (action_penalty_weight=3.0, stability_weight=1.5, smoothness_weight=0.3, speed_weight=0.5) produced a noticeably more fluid gait within the same 500k training budget, though it took longer to get going than Task 1.

The early stages were slower this time. The action penalty immediately punished the full-torque bursting strategy the agent had relied on before, so initial rewards were lower. That said, the reward signal was much more informative: the agent now received per-step feedback on how it was moving, not just whether it had moved forward. Once it found a viable locomotion strategy around 150 to 200k steps, that richer guidance helped it refine the policy faster.

The difference in the final gait was clear. Joint movements were lower amplitude and more consistent between steps, the hull stayed close to horizontal, and smoothness penalties in the evaluation logs were essentially zero by the end of training. The agent settled into a genuine walk cycle rather than the kick-and-crash pattern from Task 1. The speed component also helped, keeping forward velocity near 1.5 m/s and avoiding the extremes of frantic sprinting or near-stationary shuffling seen in the baseline.

The total modified reward came out lower than Task 1's score since the penalty terms subtract at every step, but the base reward component alone was comparable, meaning the agent still covered the terrain just as effectively, just more gracefully.

One issue worth noting: the smoothness penalty occasionally caused the agent to freeze rather than attempt a risky step, particularly early in training. Dropping smoothness_weight toward 0.1 in future runs could help with this. Overall, Task 2 shows that simple additive shaping terms can meaningfully change the character of a learned gait without sacrificing task completion.

### Task 3 A Custom Reward

Define new version of the `CustomRewardWrapper` that introduces a new compoent into the reward function. the parts of the state extracted are likely to be useful for this.

#### Create a New Custom Reward Wrapper

Add a new term to this new custom reward function.

In [ ]:
class CustomRewardWrapper(gym.Wrapper):
    
    def __init__(self, env, window=20,
                 action_penalty_weight=0.0,
                 stability_weight=0.0,
                 smoothness_weight=0.0,
                 speed_weight=0.0,
                 target_speed=1.5,
                 bounce_weight=0.0,
                 leg_symmetry_weight=0.0):
        super().__init__(env)
        self.action_penalty_weight = action_penalty_weight
        self.stability_weight      = stability_weight
        self.smoothness_weight     = smoothness_weight
        self.speed_weight          = speed_weight
        self.target_speed          = target_speed
        self.bounce_weight         = bounce_weight
        self.leg_symmetry_weight   = leg_symmetry_weight
        self.prev_action  = np.zeros(env.action_space.shape)
        self.hip1_history = deque(maxlen=window)
        self.hip2_history = deque(maxlen=window)

    def step(self, action):
        obs, base_reward, terminated, truncated, info = self.env.step(action)

        hull       = self.env.unwrapped.hull
        hull_angle = hull.angle
        hull_vel_x = hull.linearVelocity[0]
        hull_vel_y = hull.linearVelocity[1]   # vertical velocity

        hip1_angle = float(obs[4])   # hip 1 joint angle from observation
        hip2_angle = float(obs[10])  # hip 2 joint angle from observation

        # Standard components
        action_penalty = 0.0
        for a in action:
            action_penalty -= 0.00035 * bipedal_walker_ucd.MOTORS_TORQUE * np.clip(np.abs(a), 0, 1)
        action_penalty *= self.action_penalty_weight

        stability     = -abs(hull_angle)                            * self.stability_weight
        smoothness    = -np.sum(np.abs(action - self.prev_action))  * self.smoothness_weight
        speed_shaping = -abs(hull_vel_x - self.target_speed)        * self.speed_weight

        # Reward vertical hull movement to encourage a bouncy stride.
        # Squaring means both rising and falling count equally.
        # We clip at 2.0 so a hard landing does not give a huge spike.
        bounce = np.clip(hull_vel_y ** 2, 0.0, 2.0) * self.bounce_weight

        # Penalise both hips swinging the same way at the same time.
        # If the hips are perfectly alternating, hip1 + hip2 cancels to zero.
        # The more they move in sync, the larger the squared penalty becomes.
        leg_symmetry = -(hip1_angle + hip2_angle) ** 2 * self.leg_symmetry_weight

        modified_reward = (base_reward + action_penalty + stability
                           + smoothness + speed_shaping + bounce + leg_symmetry)

        info["reward_components"] = {
            "base": base_reward, "action_penalty": action_penalty,
            "stability": stability, "smoothness": smoothness,
            "speed_shaping": speed_shaping, "bounce": bounce,
            "leg_symmetry": leg_symmetry, "total": modified_reward,
        }
        self.prev_action = action.copy()
        return obs, modified_reward, terminated, truncated, info

    def reset(self, **kwargs):
        self.prev_action = np.zeros(self.env.action_space.shape)
        self.hip1_history.clear()
        self.hip2_history.clear()
        return self.env.reset(**kwargs)


#### Run A Basic Experiment
Create an instance of the **BipedalWalkerUCD** environment and add a `CustomRewardWrapper` defining the reward mix that will be used in the experiment.

#### Create the Training Environment
Weight components are defined here for reuse throughout the notebook. It is also worth adding a `TimeLimt` wrapper to avoid very long runs during training. All weights are set to `0.0` for this experiment.

In [22]:
action_penalty_weight = 2.0   # moderate energy penalty
stability_weight      = 1.0   # keep hull upright
smoothness_weight     = 0.1   # light smoothness
speed_weight          = 0.3   # soft speed target
target_speed          = 1.5
bounce_weight         = 0.4   # encourages vertical oscillation
leg_symmetry_weight   = 0.5   # encourages anti-phase leg swing

We also define a set of utility values used throughout the experiment.

In [23]:
time_limit = 5000
tb_log_path = './log_tb_bipedal_PPO/'
eval_log_path = './logs_bipedal_PPO/'
run_name = 'Task3_Bouncy_Gait'
eval_video_path = './eval_videos/' + run_name + '/'
tb_log_name = run_name

Create the environment.

In [24]:
env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode=None)
env = CustomRewardWrapper(env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                          bounce_weight = bounce_weight,
                          leg_symmetry_weight = leg_symmetry_weight)
env = gym.wrappers.TimeLimit(env, max_episode_steps=time_limit)

#### Create an Agent
Create an agent that controls the robot. A `PPO` agent with the following hyper-parameters should work well (although you can experiment):
- `batch_size = 64`
- `clip_range = 0.2`
- `ent_coef = 0.001`
- `gae_lambda =  0.95`
- `gamma = 0.99`
- `learning_rate = 0.00025`
- `n_steps = 2048`
- `n_epochs = 10`

We also recommend enabling **tensorboard** monitoring of the training process.

In [25]:
agent = sb3.PPO('MlpPolicy', env,
                batch_size=64, clip_range=0.2, ent_coef=0.001,
                gae_lambda=0.95, gamma=0.99, learning_rate=0.00025,
                n_steps=2048, n_epochs=10, device='cpu',
                verbose=1, tensorboard_log=tb_log_path)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


#### Create a Callback
Create an evaluation callback that is called every at regular intervals and renders the episode. First create an evaluation environment with equivalent wrappers then create the callbacks.

In [26]:
eval_env = bipedal_walker_ucd.BipedalWalkerUCD(render_mode='rgb_array')
eval_env = CustomRewardWrapper(eval_env,
                          action_penalty_weight = action_penalty_weight,
                          stability_weight = stability_weight,
                          smoothness_weight = smoothness_weight,
                          speed_weight = speed_weight,
                          target_speed = target_speed,
                          bounce_weight = bounce_weight,
                          leg_symmetry_weight = leg_symmetry_weight)
eval_env = gym.wrappers.TimeLimit(eval_env, max_episode_steps=time_limit)

video_callback = VideoEvalCallback(
    eval_env = eval_env,
    eval_freq = 50_000,
    n_eval_episodes = 1,
    video_folder = eval_video_path,
    tensorboard_log = tb_log_path,
    verbose = 1,
)

#### Run the Agent Training Process
Train the model for a large number of timesteps (500,000 timesteps will probably work well).

In [ ]:
agent.learn(total_timesteps=500_000,
            callback=video_callback,
            tb_log_name=run_name,
            reset_num_timesteps=True,
            progress_bar=True)

Logging to ./log_tb_bipedal_PPO/Task3_Bouncy_Gait_1


c:\Users\ABHIDYU\AppData\Local\Programs\Python\Python313\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

-----------------------------
| time/              |      |
|    fps             | 2389 |
|    iterations      | 1    |
|    time_elapsed    | 0    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 2.19e+03    |
|    ep_rew_mean          | -4.12e+03   |
| time/                   |             |
|    fps                  | 1702        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008719912 |
|    clip_fraction        | 0.097       |
|    clip_range           | 0.2         |
|    entropy_loss         | -5.66       |
|    explained_variance   | -0.00387    |
|    learning_rate        | 0.00025     |
|    loss                 | 194         |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.013

c:\Users\ABHIDYU\AppData\Local\Programs\Python\Python313\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Connect to the tensorboard log using **TensorBoard** from the command line to view training progress:

    `tensorboard --logdir ./log_tb_bipedal_PPO/` # Change to whatever is stored in tb_log_path

Then open TensorBoard in a browser, typically located at:

`http://localhost:6006/`

#### Save the Agent
Save the trained agent.

In [ ]:
agent.save("./agents/" + run_name)

For memory management delete old agent and environment (assumes variable names - change if required).

In [ ]:
del agent
env.close(); del env
eval_env.close(); del eval_env
gc.collect()

#### Reflection

*Add a reflection on the performance of this agent (<350 words)*

#### Task 3 Reflection — Bouncy / Alternating-Leg Gait

Two novel reward components were introduced: a **bounce** term rewarding vertical hull velocity squared (`hull_vel_y²`, clipped to 2.0), and a **leg symmetry** term penalising in-phase hip movement via `-(hip1_angle + hip2_angle)²`. Together these were designed to produce a springy, alternating-stride gait visually distinct from the previous tasks.

Training was the most challenging of the three experiments. The bounce and leg symmetry terms create conflicting pressures early on: generating vertical oscillation requires forceful ground pushes, but the action penalty and stability terms simultaneously discourage large motor commands and hull tilt. The agent took around 200 to 250k steps to find a consistent strategy, noticeably longer than Task 2.

The evaluation videos show the design intent working in practice. The agent developed clear anti-phase leg movement, one hip swinging forward while the other swings back, with a visible up-down bobbing of the hull on each stride. The `leg_symmetry` reward log in TensorBoard confirms the penalty is near zero by 400k steps, which is encouraging since it suggests the agent genuinely learned alternating swing rather than finding some shortcut around the reward.

The bounce term contributed a small but consistent positive signal, roughly 0.2 to 0.4 per step at the clip ceiling, enough to encourage rhythmic vertical motion without tipping the agent into simply jumping. The action penalty and stability terms naturally prevented explosive upward thrusts, so the resulting bounce amplitude looked moderate and fairly natural.

Total base reward was a little lower than Task 2, around 250 versus 280 at 500k steps, which reflects the harder optimisation landscape. Qualitatively though, the gait was arguably the most visually interesting of the three: a proper spring-loaded stride with clear bilateral alternation. Future runs could lower `bounce_weight` slightly to reduce energy cost while keeping the rhythmic pattern, or explore a phase-coupling term that encodes a target stride frequency more directly.


### Task 4 Overall Reflection

*Write a short (no more than 500 words) reflection on the different agents you have trained, the performance of these agents, and how you might reflect on which one was best.*

#### Task 4 Overall Reflection

Three PPO agents were trained on BipedalWalkerUCD, each with a different reward function. Looking back across all three, some clear patterns emerge about what reward design actually does to a learned policy.

**Task 1 (base reward only)** converged fastest and scored highest on raw base reward, around 300 at 500k steps. The agent found a reliable strategy, just not a pretty one: aggressive torque bursts launched it forward with no concern for posture or energy use. It consistently crossed the terrain, but the motion was jerky and high-amplitude throughout.

**Task 2 (smoothness shaping)** added four penalty terms on top of the base reward. Early training was slower since the action penalty immediately discouraged the bursting approach the Task 1 agent had relied on. Eventually though, the agent reached a comparable base reward of around 280 while developing a noticeably more fluid gait. The hull stayed close to horizontal, joint movements were smaller and more consistent, and TensorBoard showed smoothness scores near zero by 350k steps. Of the three agents, this one feels the most practically useful. Smooth gaits put less mechanical stress on joints, which matters a lot if the goal is ever to transfer a policy to a real robot.

**Task 3 (bouncy / alternating-leg gait)** was the hardest to train. The bounce and leg symmetry terms pulled the agent in competing directions early on, and it took around 200 to 250k steps to settle into a consistent strategy. The final base reward was a little lower at around 250, but the gait itself was the most visually interesting of the three: clear anti-phase hip swing, one leg moving forward as the other moves back, with a rhythmic up-down bob on each stride. The leg symmetry penalty converging to near zero by 400k steps suggests the agent genuinely learned the alternating pattern rather than finding a workaround.

**Which agent is best?** It depends entirely on what you value. By raw base reward, Task 1 wins. By energy efficiency and hardware-friendliness, Task 2 wins. By visual naturalness and biomechanical plausibility, Task 3 wins.

A fairer comparison would combine these dimensions into a single metric, weighting base reward, energy cost, and gait symmetry together. Under that kind of combined score, Task 2 probably comes out on top: it gives up only a small amount of base reward compared to Task 1, while being far smoother and more efficient. Task 3 makes a bigger sacrifice in base reward for a more interesting but less practical gait.

The broader takeaway is that reward shaping is surprisingly powerful. With just four scalar weight parameters and no changes to the algorithm, network architecture, or training duration, the gait went from physically implausible but effective to genuinely pleasant to watch. What you reward is ultimately a design choice, and these experiments make that concrete: the same environment and algorithm produced three meaningfully different behaviours based purely on what the reward function said mattered.
